# Data Profiling

##### Librerias

In [152]:
import os
from pathlib import Path
import polars as pl

#### 1. Data Profiling Pacientes

In [153]:
# Con la finalidad de no perder algun dato en la lectura, se tratara a todos como String para proteger la data corrupta
# y evitar pérdidas por casteo automático erróneo.
path_pacientes = (Path(os.getcwd())).parent / "datos" / "pacientes.csv"
ctx_pacientes = pl.read_csv(
    path_pacientes,
    separator=";",
    encoding="iso-8859-1",
    infer_schema_length=0,  # Fuerza a Polars a tratar todo como String inicialmente
    null_values=["null", "NULL", ""]  # Normaliza los textos vacíos y nulos de inmediato a null real
)

In [154]:
ctx_pacientes.head(5)

id,nombre,apellido,dni,telefono,email,fecha_nacimiento,sexo,direccion,created_at
str,str,str,str,str,str,str,str,str,str
"""pac_000001""",null,"""García""","""12345678""","""123""","""@invalid..com""","""1987/13/45""","""X""","""Calle Libertad 4112""","""2025-02-18T03:15:52.929338"""
"""pac_000002""","""Luis""","""Martínez""","""12345678""","""+54 9 9935-2424""","""luis@example.com""","""1994-01-01""","""M""","""Calle Secundaria 3911""","""2023-07-22T03:15:52.930333"""
"""pac_000003""","""Juann""","""García""","""12345678""","""+54 9 4257-9928""","""juann@example.com""","""1993-04-15""","""F""","""Calle Principal 2715""","""2022-06-22T03:15:52.930333"""
"""pac_000004""","""Isabel""","""Morales""","""12345678""","""+54 9 5552-3547""","""isabel@example.com""","""1967-06-04""","""M""","""Calle Paz 1684""","""2024-05-16T03:15:52.930333"""
"""pac_000005""","""Teresa""","""Cruz""","""12345678""","""+54 9 5333-1711""","""teresa@example.com""","""1998-09-04""","""F""","""Calle Principal 9144""","""2024-09-28T03:15:52.930333"""


### A validacion manual todos los registros que cumplen las siguientes reglas:
- Regla 1 - DNI esta en blanco o vacio 
- Regla 2 - DNI contiene texto
- Regla 3 - DNI hace referencia a mas de un registro. 

In [155]:
# Reglas 1 y 2
regex_solo_numeros = r"^\d+$"
pacientes_por_validar_r1_r2  = ctx_pacientes.filter(
pl.col("dni").is_null() | (pl.col("dni").str.strip_chars() == "") | # Regla 1
pl.col("dni").str.contains(regex_solo_numeros).eq(False)            # Regla 2
).with_columns(
    pl.lit("Campo_DNI_Blanco_Vacio").alias("motivo_rechazo")     # Detalle motivo del rechazo para la auditoría técnica
)
ctx_pacientes_limpio = ctx_pacientes.filter( ~ ( 
    pl.col("dni").is_null() | (pl.col("dni").str.strip_chars() == "") | 
    pl.col("dni").str.contains(regex_solo_numeros).eq(False)
    )
)

In [156]:
# Raglas 3
dnis_repetidos_lista = (ctx_pacientes_limpio.group_by("dni").agg(pl.len().alias("conteo"))
                       .filter(pl.col("conteo") > 1)
                       .select("dni").to_series().to_list()
)

pacientes_por_validar_r3 = (ctx_pacientes_limpio.filter(pl.col("dni").is_in(dnis_repetidos_lista))
                                                .with_columns(pl.lit("Campo_DNI_Duplicado").alias("motivo_rechazo"))
)

ctx_pacientes_limpio = ctx_pacientes_limpio.filter(~pl.col("id").is_in(pacientes_por_validar_r3["id"].to_list())
)

### Limpiar los datos trash

#### Limpiar Telefonos

In [157]:
# Reemplazamos por None los telefonos que contengan letras
ctx_pacientes_limpio = ctx_pacientes_limpio.with_columns(pl.when(pl.col("telefono").is_not_null() & 
                                                                 pl.col("telefono").str.contains(r"[a-zA-Z]") 
                                                                )
                                                           .then(None) # Borra el dato
                                                           .otherwise(pl.col("telefono"))
                                                           .alias("telefono")
                                                        )
# 2. Reemplazamos por None los que no cumplan la condición
DIGITOS_MINIMOS = 7
DIGITOS_MAXIMOS = 15
ctx_pacientes_limpio = ctx_pacientes_limpio.with_columns(pl.when( pl.col("telefono").is_not_null() & 
                                                                 (pl.col("telefono").str.len_chars() >= DIGITOS_MINIMOS) & 
                                                                 (pl.col("telefono").str.len_chars() <= DIGITOS_MAXIMOS)
                                                                )
                                                            .then(pl.col("telefono"))
                                                            .otherwise(None)
                                                            .alias("telefono")
                                                        )                                                      

#### Limpiar email

In [158]:
# 1. Definir el patrón estructural del correo electrónico
regex_email_valido = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

# 2. Reemplazar con None si no contiene la estructura válida
ctx_pacientes_limpio = ctx_pacientes_limpio.with_columns(pl.when(pl.col("email").is_not_null() & pl.col("email").str.contains(regex_email_valido)
                                                                )
                                                            .then(pl.col("email"))
                                                            .otherwise(None)        
                                                            .alias("email")
                                                        )

#### Limpiar Fecha Nacimiento

In [159]:
ctx_pacientes_limpio = ctx_pacientes_limpio.with_columns(pl.when(pl.coalesce([pl.col("fecha_nacimiento").str.to_date(format="%Y-%m-%d", strict=False),
                                                                              pl.col("fecha_nacimiento").str.to_date(format="%d/%m/%Y", strict=False)
                                                                             ]).is_not_null()
                                                                 )
                                                           .then(pl.col("fecha_nacimiento")) 
                                                           .otherwise(None)         
                                                           .alias("fecha_nacimiento")
                                                         )

#### Limpiar campo Sexo

In [160]:
VALORES_SEXO_VALIDOS = ["F", "M", "X"]

# 2. Convertimos a mayúsculas y aplicamos el filtro condicional de blanqueo
ctx_pacientes_limpio = ctx_pacientes_limpio.with_columns(pl.col("sexo").str.strip_chars().str.to_uppercase().alias("sexo")
                                          ).with_columns(pl.when(pl.col("sexo").is_in(VALORES_SEXO_VALIDOS))
                                                           .then(pl.col("sexo"))
                                                           .otherwise(None)
                                                           .alias("sexo")
                                                        )

#### Consolidad Paciente a Validar

In [161]:
pacientes_por_validar = pl.concat(
    [pacientes_por_validar_r1_r2, pacientes_por_validar_r3],
    how="diagonal"
)

In [162]:
pacientes_por_validar.count()

id,nombre,apellido,dni,telefono,email,fecha_nacimiento,sexo,direccion,created_at,motivo_rechazo
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
20,19,20,15,20,20,20,20,20,20,20


In [163]:
ctx_pacientes.count()

id,nombre,apellido,dni,telefono,email,fecha_nacimiento,sexo,direccion,created_at
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
255,245,255,250,247,245,249,237,255,255


In [164]:
ctx_pacientes_limpio.count()

id,nombre,apellido,dni,telefono,email,fecha_nacimiento,sexo,direccion,created_at
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
235,226,235,235,212,202,218,217,235,235


### Auditoría de Cambios línea por línea.

In [165]:
columnas_a_auditar = ["telefono", "email", "fecha_nacimiento", "sexo"]

ctx_pacientes_tmp = ctx_pacientes.select( ["id"] + columnas_a_auditar
                                                        ).rename({col: f"{col}_orig" for col in columnas_a_auditar})

comparativa_raw = ctx_pacientes_limpio.join(ctx_pacientes_tmp, on="id", how="inner")



In [166]:
# 1. Inicializamos la condición de cambio en False
condicion_hubo_cambio = pl.lit(False)

# 2. Recorremos los 4 campos aplicando un ajuste de tipo para fechas si es necesario
for col in columnas_a_auditar:
    col_limpia = pl.col(col)
    col_original = pl.col(f"{col}_orig")
        
    # Ahora la comparación es 100% segura entre tipos idénticos
    condicion_hubo_cambio = condicion_hubo_cambio | col_limpia.ne_missing(col_original)

# 3. Filtramos el dataset para quedarnos SOLO con las filas modificadas
pacientes_modificados = comparativa_raw.filter(condicion_hubo_cambio)

# 4. Construimos la lista de columnas organizada para el reporte final
columnas_reporte = ["id", "nombre"]
for col in columnas_a_auditar:
    columnas_reporte.extend([f"{col}_orig", col])

# 5. Creamos el DataFrame final con el antes y el después ordenado
reporte_cambios_detallado = pacientes_modificados.select(columnas_reporte)

In [167]:
reporte_cambios_detallado.count()

id,nombre,telefono_orig,telefono,email_orig,email,fecha_nacimiento_orig,fecha_nacimiento,sexo_orig,sexo
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
41,36,41,26,41,18,35,24,40,40


In [170]:
reporte_cambios_detallado.head(40)

id,nombre,telefono_orig,telefono,email_orig,email,fecha_nacimiento_orig,fecha_nacimiento,sexo_orig,sexo
str,str,str,str,str,str,str,str,str,str
"""pac_000022""","""Isabel""","""+54 9 8651-1887""","""+54 9 8651-1887""","""Isabelinvalid.com""",null,"""1952-01-13""","""1952-01-13""","""F""","""F"""
"""pac_000031""","""María""","""123""",null,"""maría@example.com""",null,"""1998-10-19""","""1998-10-19""","""M""","""M"""
"""pac_000033""","""Diego""","""+54 11 1234-5678""",null,"""diego@example.com""","""diego@example.com""","""1978-10-26""","""1978-10-26""","""M""","""M"""
"""pac_000041""","""Rosa""","""+54 9 1387-4164""","""+54 9 1387-4164""","""Rosa@invalid..com""","""Rosa@invalid..com""","""1987/13/45""",null,"""F""","""F"""
"""pac_000042""","""María""","""+54 9 6753-9346""","""+54 9 6753-9346""","""maría@example.com""",null,"""2025-01-01""","""2025-01-01""","""F""","""F"""
…,…,…,…,…,…,…,…,…,…
"""pac_000216""","""Pilar""","""+54 9 7666-8793""","""+54 9 7666-8793""","""pilar@example.com""","""pilar@example.com""","""not-a-date""",null,"""F""","""F"""
"""pac_000225""","""Luis""","""+54 11 1234-5678""",null,"""luis@example.com""","""luis@example.com""","""1959-08-24""","""1959-08-24""","""F""","""F"""
"""pac_000232""","""Javier""","""+54 9 2314-9835""","""+54 9 2314-9835""","""Javierinvalid.com""",null,"""1957-11-28""","""1957-11-28""","""F""","""F"""
